In [1]:
import sys
from pathlib import Path

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent
sys.path.insert(0, str(raiz))

from lakehouse import Lakehouse
from analisis_comun import (FILTRO, GRUPO_EDAD, GRUPO_GESTACION, GRUPO_ALTITUD, GRUPO_POBREZA,
                            ORDEN_EDAD, ORDEN_GESTACION, ORDEN_EDUCACION, ORDEN_ALTITUD, ORDEN_POBREZA)

db = Lakehouse()
consultar = db.consultar
ejecutar = db.ejecutar

import pandas as pd

resultados = raiz / 'resultados'
resultados.mkdir(exist_ok=True)
pd.set_option('display.width', 200)

In [2]:
# 1. Flujo de selección de la población de análisis
PERIODO = "anio_nacimiento BETWEEN 2015 AND 2025"
pasos = {
    'Registros en el archivo': 'TRUE',
    'Nacidos en 2015-2025': PERIODO,
    'Partos únicos': f'{PERIODO} AND es_parto_unico',
    'Peso entre 500 y 6000 g': f'{PERIODO} AND es_parto_unico AND es_peso_valido',
    'Gestación de 22 a 44 semanas': f'{PERIODO} AND es_parto_unico AND es_peso_valido AND es_gestacion_valida',
    'Peso coherente con la gestación': f'{PERIODO} AND es_parto_unico AND es_peso_valido AND es_gestacion_valida '
                                       'AND es_peso_plausible_gestacion',
}
columnas = ',\n       '.join(f'COUNT_IF({condicion}) AS paso_{i}' for i, condicion in enumerate(pasos.values()))
conteos = consultar(f"""
SELECT {columnas},
       COUNT_IF({FILTRO}) AS poblacion_analisis
FROM __CATALOGO__.curated.nacimientos_peru
""")
assert int(conteos.loc[0, f'paso_{len(pasos) - 1}']) == int(conteos.loc[0, 'poblacion_analisis'])
flujo = pd.DataFrame({'paso': list(pasos),
                      'registros': [int(conteos.loc[0, f'paso_{i}']) for i in range(len(pasos))]})
flujo['excluidos'] = (-flujo['registros'].diff()).fillna(0).astype(int)
flujo['%_excluidos_del_paso_anterior'] = (100 * flujo['excluidos'] / flujo['registros'].shift()).round(2)
print(flujo.to_string(index=False))
flujo.to_csv(resultados / 'flujo_seleccion.csv', index=False)

                           paso  registros  excluidos  %_excluidos_del_paso_anterior
        Registros en el archivo    5095731          0                            NaN
           Nacidos en 2015-2025    4904829     190902                           3.75
                  Partos únicos    4811593      93236                           1.90
        Peso entre 500 y 6000 g    4808635       2958                           0.06
   Gestación de 22 a 44 semanas    4808464        171                           0.00
Peso coherente con la gestación    4805234       3230                           0.07


In [3]:
# 2. Completitud de las variables en la población de análisis
sin_dato = {
    'Edad de la madre (fuera de 10-60 o vacía)': 'NOT es_edad_madre_valida',
    'Sexo del recién nacido': 'sexo_nacido IS NULL',
    'Talla (fuera de 30-60 cm o vacía)': 'NOT es_talla_valida',
    'Nivel de instrucción': 'nivel_instruccion_madre IS NULL',
    'Estado civil': 'estado_civil IS NULL',
    'Embarazos previos': 'num_embar_madre_cat IS NULL',
    'Hijos vivos': 'hijos_vivos_madre_cat IS NULL',
    'Abortos o nonatos': 'antecedentes_abortos_cat IS NULL',
    'País de la madre': 'pais_madre IS NULL',
    'Ubigeo de residencia': 'id_ubigeo_inei IS NULL',
    'Contexto del distrito (altitud, IDH, pobreza)': 'altitud_msnm IS NULL',
    'Establecimiento (IPRESS)': 'codigo_ipress IS NULL',
    'Lugar del parto': 'lugar_nacido IS NULL',
    'Quién atiende el parto': 'atiende_parto IS NULL',
    'Financiador del parto': 'financiador_parto IS NULL',
    # Valores registrados pero poco confiables
    'Hijos fallecidos = -1 (0 o ignorado)': "hijos_fallecidos_madre_cat = '0 O IGNORADO'",
    'Estado civil = SOLTERO (posible registro por defecto)': "estado_civil = 'SOLTERO'",
}
columnas = ',\n       '.join(f'ROUND(100 * AVG(CAST(COALESCE({condicion}, FALSE) AS INT)), 2) AS v{i}'
                               for i, condicion in enumerate(sin_dato.values()))
porcentajes = consultar(f"""
SELECT {columnas}
FROM __CATALOGO__.curated.nacimientos_peru
WHERE {FILTRO}
""")
completitud = pd.DataFrame({'variable': list(sin_dato),
                            '%_registros': [float(porcentajes.loc[0, f'v{i}']) for i in range(len(sin_dato))]})
print(completitud.to_string(index=False))
completitud.to_csv(resultados / 'completitud_variables.csv', index=False)

                                             variable  %_registros
            Edad de la madre (fuera de 10-60 o vacía)         0.00
                               Sexo del recién nacido         0.00
                    Talla (fuera de 30-60 cm o vacía)         0.12
                                 Nivel de instrucción         0.05
                                         Estado civil         4.18
                                    Embarazos previos         1.03
                                          Hijos vivos         0.77
                                    Abortos o nonatos         8.54
                                     País de la madre         0.00
                                 Ubigeo de residencia         0.18
        Contexto del distrito (altitud, IDH, pobreza)         0.25
                             Establecimiento (IPRESS)         0.00
                                      Lugar del parto         0.00
                               Quién atiende el parto         

In [4]:
# 3. Resumen de variables numéricas: todos, con bajo peso y con peso ≥2500 g
numericas = {
    'Peso al nacer (g)': 'peso_nacido_g',
    'Talla (cm)': 'CASE WHEN es_talla_valida THEN talla_nacido_cm END',
    'Semanas de gestación': 'duracion_embarazo_semanas',
    'Edad de la madre (años)': 'CASE WHEN es_edad_madre_valida THEN edad_madre END',
    'Altitud del distrito (msnm)': 'altitud_msnm',
    'IDH 2019 del distrito': 'idh_2019',
    'Pobreza del distrito (%)': 'pobreza_pct',
}
consultas = [f"""
SELECT '{etiqueta}' AS variable,
       CASE WHEN GROUPING(es_bajo_peso_nacer) = 1 THEN 'Todos'
            WHEN es_bajo_peso_nacer THEN 'Bajo peso (<2500 g)'
            ELSE 'Peso ≥2500 g' END AS grupo,
       COUNT({expresion}) AS n,
       AVG({expresion}) AS media,
       STDDEV({expresion}) AS desviacion,
       percentile_approx({expresion}, ARRAY(0.05, 0.25, 0.5, 0.75, 0.95)) AS percentiles
FROM __CATALOGO__.curated.nacimientos_peru
WHERE {FILTRO}
GROUP BY GROUPING SETS ((), (es_bajo_peso_nacer))""" for etiqueta, expresion in numericas.items()]
resumen = consultar('\nUNION ALL\n'.join(consultas))
percentiles = pd.DataFrame(resumen.pop('percentiles').tolist(), columns=['p5', 'p25', 'mediana', 'p75', 'p95'])
resumen = pd.concat([resumen, percentiles], axis=1)
resumen['grupo'] = pd.Categorical(resumen['grupo'], ['Todos', 'Bajo peso (<2500 g)', 'Peso ≥2500 g'])
resumen['variable'] = pd.Categorical(resumen['variable'], list(numericas))
resumen = resumen.sort_values(['variable', 'grupo'])
print(resumen.round(2).to_string(index=False))
resumen.to_csv(resultados / 'resumen_numerico.csv', index=False)

                   variable               grupo       n   media  desviacion      p5     p25  mediana     p75     p95
          Peso al nacer (g)               Todos 4805234 3265.66      505.97 2460.00 2990.00  3280.00 3580.00 4040.00
          Peso al nacer (g) Bajo peso (<2500 g)  261849 2057.58      446.58 1030.00 1880.00  2220.00 2390.00 2475.00
          Peso al nacer (g)        Peso ≥2500 g 4543385 3335.28      412.68 2695.00 3040.00  3310.00 3600.00 4050.00
                 Talla (cm)               Todos 4799541   49.23        2.40   45.50   48.00    49.50   50.80   52.00
                 Talla (cm) Bajo peso (<2500 g)  259753   43.95        3.75   36.00   42.50    45.00   46.00   48.00
                 Talla (cm)        Peso ≥2500 g 4539788   49.53        1.90   46.50   48.30    50.00   51.00   52.50
       Semanas de gestación               Todos 4805234   38.68        1.63   36.00   38.00    39.00   40.00   40.00
       Semanas de gestación Bajo peso (<2500 g)  261849   35.14 

In [5]:
# 4. Tabla 1: características de la población según bajo peso al nacer
dimensiones = {
    'Edad de la madre': GRUPO_EDAD,
    'Nivel de instrucción': "COALESCE(nivel_instruccion_madre, 'SIN DATO')",
    'Estado civil': "COALESCE(estado_civil, 'SIN DATO')",
    'Hijos vivos': "COALESCE(hijos_vivos_madre_cat, 'SIN DATO')",
    'Embarazos': "COALESCE(num_embar_madre_cat, 'SIN DATO')",
    'Financiador del parto': "COALESCE(financiador_parto, 'SIN DATO')",
    'País de la madre': "COALESCE(pais_madre_grupo, 'SIN DATO')",
    'Sexo del recién nacido': "COALESCE(sexo_nacido, 'SIN DATO')",
    'Semanas de gestación': GRUPO_GESTACION,
    'Macrorregión de residencia': "COALESCE(macroregion, 'SIN DATO')",
    'Altitud del distrito': GRUPO_ALTITUD,
    'Pobreza del distrito': GRUPO_POBREZA,
    'Lugar del parto': "COALESCE(lugar_nacido, 'SIN DATO')",
}
estructuras = ',\n  '.join(f"NAMED_STRUCT('variable', '{nombre}', 'valor', {expresion})"
                             for nombre, expresion in dimensiones.items())
ejecutar(f"""CREATE OR REPLACE TABLE __CATALOGO__.analytics.tabla1_caracteristicas
USING DELTA AS
WITH conteos AS (
  SELECT dim.variable, dim.valor,
         COUNT(*) AS n,
         COUNT_IF(es_bajo_peso_nacer) AS bajo_peso,
         COUNT_IF(NOT es_bajo_peso_nacer) AS peso_normal
  FROM __CATALOGO__.curated.nacimientos_peru
  LATERAL VIEW EXPLODE(ARRAY(
  {estructuras}
  )) e AS dim
  WHERE {FILTRO}
  GROUP BY dim.variable, dim.valor
)
SELECT variable, valor,
       n, ROUND(100 * n / SUM(n) OVER (PARTITION BY variable), 1) AS pct_total,
       bajo_peso, ROUND(100 * bajo_peso / SUM(bajo_peso) OVER (PARTITION BY variable), 1) AS pct_en_bajo_peso,
       peso_normal, ROUND(100 * peso_normal / SUM(peso_normal) OVER (PARTITION BY variable), 1) AS pct_en_peso_normal,
       ROUND(100 * bajo_peso / n, 2) AS porcentaje_bajo_peso
FROM conteos""")
tabla1 = consultar('SELECT * FROM __CATALOGO__.analytics.tabla1_caracteristicas')
ordenes = {'Edad de la madre': ORDEN_EDAD, 'Nivel de instrucción': ORDEN_EDUCACION,
           'Semanas de gestación': ORDEN_GESTACION, 'Altitud del distrito': ORDEN_ALTITUD,
           'Pobreza del distrito': ORDEN_POBREZA, 'Hijos vivos': ['1', '2', '3', '4', '>=5'],
           'Embarazos': ['1', '2', '3', '4', '>=5']}


def orden_filas(fila):
    orden = ordenes.get(fila['variable'], [])
    posicion = orden.index(fila['valor']) if fila['valor'] in orden else len(orden) + 1_000_000 - fila['n']
    return (list(dimensiones).index(fila['variable']), fila['valor'] == 'SIN DATO', posicion)


tabla1 = tabla1.loc[sorted(tabla1.index, key=lambda i: orden_filas(tabla1.loc[i]))]
presentacion = pd.DataFrame({
    'Variable': tabla1['variable'], 'Categoría': tabla1['valor'],
    'Total n (%)': [f'{n:,} ({p:.1f})' for n, p in zip(tabla1['n'], tabla1['pct_total'])],
    'Bajo peso n (%)': [f'{n:,} ({p:.1f})' for n, p in zip(tabla1['bajo_peso'], tabla1['pct_en_bajo_peso'])],
    'Peso ≥2500 g n (%)': [f'{n:,} ({p:.1f})' for n, p in zip(tabla1['peso_normal'], tabla1['pct_en_peso_normal'])],
    '% bajo peso': tabla1['porcentaje_bajo_peso'],
})
print(presentacion.to_string(index=False))
presentacion.to_csv(resultados / 'tabla1_caracteristicas.csv', index=False)

                  Variable                    Categoría      Total n (%) Bajo peso n (%) Peso ≥2500 g n (%)  % bajo peso
          Edad de la madre                        10-14     24,989 (0.5)     2,885 (1.1)       22,104 (0.5)        11.55
          Edad de la madre                        15-19   615,430 (12.8)   43,961 (16.8)     571,469 (12.6)         7.14
          Edad de la madre                        20-34 3,258,293 (67.8)  157,627 (60.2)   3,100,666 (68.2)         4.84
          Edad de la madre                        35-39   670,852 (14.0)   39,303 (15.0)     631,549 (13.9)         5.86
          Edad de la madre                        40-44    220,995 (4.6)    16,457 (6.3)      204,538 (4.5)         7.45
          Edad de la madre                        45-60     14,616 (0.3)     1,611 (0.6)       13,005 (0.3)        11.02
          Edad de la madre                     SIN DATO         59 (0.0)         5 (0.0)           54 (0.0)         8.47
      Nivel de instrucción      

In [6]:
# 5. Resumen en texto para la sección de métodos y resultados
poblacion = int(flujo['registros'].iloc[-1])
total = int(flujo['registros'].iloc[0])
peso = resumen[(resumen['variable'] == 'Peso al nacer (g)') & (resumen['grupo'] == 'Todos')].iloc[0]
bajo = int(tabla1.loc[tabla1['variable'] == 'Edad de la madre', 'bajo_peso'].sum())
prematuros = tabla1[(tabla1['variable'] == 'Semanas de gestación') &
                    tabla1['valor'].isin(ORDEN_GESTACION[:4])]['n'].sum()
print(f'De {total:,} registros del CNV se analizaron {poblacion:,} partos únicos de 2015-2025 '
      f'({100 * poblacion / total:.1f} %).')
print(f'Peso al nacer: mediana {peso["mediana"]:.0f} g (RIC {peso["p25"]:.0f}-{peso["p75"]:.0f}); '
      f'bajo peso {bajo:,} ({100 * bajo / poblacion:.2f} %); prematuros {100 * prematuros / poblacion:.2f} %.')
poco_confiables = completitud[completitud['%_registros'] > 5]
print('Variables con más de 5 % sin dato o poco confiables:')
print(poco_confiables.to_string(index=False))
print('Tablas guardadas en', resultados.relative_to(raiz))

De 5,095,731 registros del CNV se analizaron 4,805,234 partos únicos de 2015-2025 (94.3 %).
Peso al nacer: mediana 3280 g (RIC 2990-3580); bajo peso 261,849 (5.45 %); prematuros 5.96 %.
Variables con más de 5 % sin dato o poco confiables:
                                             variable  %_registros
                                    Abortos o nonatos         8.54
                 Hijos fallecidos = -1 (0 o ignorado)        96.99
Estado civil = SOLTERO (posible registro por defecto)        84.66
Tablas guardadas en resultados
